### Imports

In [ ]:
import os
import torch
import torchvision.transforms as T
from torch import optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

import sys
sys.path.append('..')

from src.utils.dataloading import ClipDataset
from src.utils.model import MultitaskLoss, AgeNet
from src.utils.utils import train

# Modelling

We build the model in this section. The task is to estimate a person’s age while also detecting whether a person is present in the image or not. We use ``EfficientNet`` as the backbone for feature extraction, followed by a Transformer layer to model temporal information across frames. The network has two output heads: one for binary classification of person presence, and another for age regression.

## Setup

In [ ]:
# setup
seed = 42
torch.manual_seed(seed)
# device setup
device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
print(f"Training on device {device}.")

## Preprocessing

### reading and splitting
The dataset is divided into training, validation, and test sets using an 80–10–10 split, respectively.

In [ ]:
path = "../data/dataset"
files = [os.path.join(path, f) for f in os.listdir(path)]

train_files, test_files = train_test_split(
    files,
    test_size=0.2,
    random_state=seed,
    shuffle=True
)

val_files, test_files = train_test_split(
    test_files,
    test_size=0.5,
    random_state=seed,
    shuffle=True
)

### normalizing
For normalization, we can either compute the dataset statistics ourselves, or use the precomputed ImageNet statistics since we are gonna use ``EfficientNet`` as the backbone model.

In [ ]:
# computing statistics per channel (rgb)
train_mean = torch.zeros(3)
train_std  = torch.zeros(3)

# precomputed mean and std
train_mean = torch.tensor([0.485, 0.456, 0.406])
train_std  = torch.tensor([0.229, 0.224, 0.225])

# computing the statistics for our data set
# for file in train_files:
#     sample = torch.load(file)
#     frames = sample["frames"]

#     mean += frames.mean(dim=(0, 2, 3))
#     std  += frames.std(dim=(0, 2, 3))

In [ ]:
# normalization transform
norm_transform = T.Normalize(mean=train_mean, std=train_std)

We are not gonna transform and rewrite the data directly. Instead, we pass the transformation as an argument to the dataset class, where it will be applied in the ``__getitem__`` method.

### loaders

In [ ]:
# create the datasets
train_dataset = ClipDataset(train_files, transform=norm_transform)
val_dataset   = ClipDataset(val_files, transform=norm_transform)
test_dataset  = ClipDataset(test_files, transform=norm_transform)

# create the dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

## Architecture

In [ ]:
model = AgeNet()
model.freeze_backbone()

## Training

### loss function

For the loss function we use the `MultitaskLoss` implemented in `model.py`. Its essentially a combination of binary cross entropy loss and mean squared error.

In [ ]:
criterion = MultitaskLoss()

### optimizer

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)

### train

In [ ]:
loaders = {
    "train": train_loader,
    "val": val_loader
}

losses = train(
    model=model,
    loaders=loaders,
    criterion=criterion,
    optimizer=optimizer,
    epochs=30,
    device=device,
    save_path="../data/model/AgeNet"
)